# Tier 2 — Matched-Event Framing, **Hardened v2** (N=10)

Implements `TIER2_HARDENING.md`. Makes the matched-event null *trustworthy*:
1. **Validated, upgraded measures** (gold set; ensemble sentiment, dependency-parse attribution, UD agency).
2. **Powered** (cluster the FULL pool; up-front power analysis).
3. **Hardened clustering** (calibrated threshold + NER-overlap + Leiden/HDBSCAN + purity) and **statistics**
   (mixed-effects `measure ~ outlet + char_len + (1|event)`; pre-specified confirmatory family vs exploratory FDR).

Constraint: **no generative LLMs** (encoders, fine-tuned classifiers, parsers, lexicons, classical stats only).
Run top-to-bottom in Colab/GPU. Cells gate on the previous step; read the printed verdicts before trusting any null.

## 0 · Install (all non-generative)

In [ ]:
!pip -q install sentence-transformers transformers sentencepiece scikit-learn statsmodels pandas pyarrow seaborn
# clustering + parsing
!pip -q install hdbscan leidenalg python-igraph 2>/dev/null || echo "leiden/hdbscan optional - greedy fallback available"
!python -m spacy download fr_core_news_lg -q
print("installs done")

## 1 · Config

In [ ]:
import os, sys, glob, json, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore')

# --- paths: SAME convention as tier1_tone_experiment.ipynb / tier2_matched_event_framing.ipynb ---
EDITORIAL_DIR = os.environ.get('EDITORIAL_DIR', '.')   # package root by default; override to your rebuilt-corpus dir
PANEL = 'n10'
sys.path.insert(0, EDITORIAL_DIR)   # for split_utils, gen_*.py
OUT = os.path.join(EDITORIAL_DIR, f'analysis_{PANEL}', 'tier2_v2'); os.makedirs(OUT, exist_ok=True)

# locate the balanced sampled corpus (pilot population) by glob, like the working notebooks
_cands = ([os.path.join(EDITORIAL_DIR, f'results/{PANEL}/sampled_dataset_{PANEL}.parquet'),
           os.path.join(EDITORIAL_DIR, f'results/{PANEL}/sampled_dataset.parquet')]
          + glob.glob(os.path.join(EDITORIAL_DIR, f'**/sampled_dataset*{PANEL}*.parquet'), recursive=True)
          + glob.glob(os.path.join(EDITORIAL_DIR, '**/sampled_dataset*.parquet'), recursive=True))
DATA_PATH = next((p for p in _cands if os.path.exists(p)), None)
assert DATA_PATH, 'Could not find sampled_dataset parquet under EDITORIAL_DIR — set DATA_PATH manually.'

# full article pool (Part 2 powering run) — phase1 caches it as articles_pool.parquet under BASE_DIR
_pool = ([os.path.join(EDITORIAL_DIR, 'articles_pool.parquet')]
         + glob.glob(os.path.join(EDITORIAL_DIR, '**/articles_pool.parquet'), recursive=True))
FULL_POOL = next((p for p in _pool if os.path.exists(p)), None)

# --- knobs (specify before running) ---
USE_FULL_POOL   = False     # Part 2 lever: True -> cluster the full article pool, not the balanced subset
RECOMPUTE_EMB   = False
DATE_WINDOW     = 2         # +/- days for same-event candidacy
COS_THRESHOLD   = 0.70      # calibrated in step 4b; sweep 0.65/0.70/0.75
MIN_OUTLETS     = 4         # distinct outlets to call a cluster a 'matched event'
NER_OVERLAP_MIN = 1         # shared PER/LOC/ORG required to join a cluster (0 = off)
ENCODER         = 'dangvantuan/sentence-camembert-large'   # fallback: paraphrase-multilingual-mpnet-base-v2

# measure admission gates (Part 1b) - a measure enters the CONFIRMATORY family only if it clears these vs gold
GATE_KAPPA = 0.60   # binary measures (attribution, passive, hedging)
GATE_RHO   = 0.50   # continuous measures (valence)

# reference/aliased outlet for the mixed-effects sum-to-zero coding (must be NON-confirmatory & present in panel)
REF_OUTLET = 'Le Monde'

# the paper's 10-outlet panel (used to filter the pool if it's a wider superset)
PANEL_OUTLETS = ['BFMTV','Franceinfo','La Croix','Le Figaro','Le Monde','Le Parisien','Les Echos','Libération','Ouest-France','TF1 INFO']

# pre-specified CONFIRMATORY predictions (directional, from Tiers 0/1). 'lt'/'gt' = one-sided.
CONFIRMATORY = [
    ('Libération', 'valence',     'lt'),   # Libération more negative on the same event
    ('Les Echos',  'attribution', 'lt'),   # Les Échos attributes least
    ('BFMTV',      'attribution', 'gt'),   # BFMTV attributes more
    ('BFMTV',      'agent_subject','gt'),  # BFMTV more agentive subjects
]
print('EDITORIAL_DIR:', EDITORIAL_DIR)
print('data :', DATA_PATH)
print('out  :', OUT)
print('full pool:', FULL_POOL or 'NOT FOUND (only needed when USE_FULL_POOL=True)')
print('USE_FULL_POOL =', USE_FULL_POOL)

## 2 · Load data

In [ ]:
import html, unicodedata, hashlib
if USE_FULL_POOL:
    assert FULL_POOL, 'USE_FULL_POOL=True but articles_pool.parquet not found under EDITORIAL_DIR'
    src = FULL_POOL
else:
    src = DATA_PATH
print('loading:', src)
df = pd.read_parquet(src)
print('columns in source:', list(df.columns))

# normalize required column names (pool schema differs from the balanced sample)
def _pick(df, names):
    for n in names:
        if n in df.columns: return n
    return None
col_id   = _pick(df, ['article_id','id','doc_id','article','uid'])
col_title= _pick(df, ['title','headline','titre','text'])
col_out  = _pick(df, ['outlet','source','media','journal'])
col_cat  = _pick(df, ['category','cat','desk','rubrique','section'])
col_dt   = _pick(df, ['dt','date','published','published_at','pub_date','datetime','timestamp'])
missing=[k for k,v in dict(article_id=col_id,title=col_title,outlet=col_out,category=col_cat,dt=col_dt).items() if v is None]
assert not missing, f'pool missing required field(s) {missing}; columns = {list(df.columns)}'
df = df.rename(columns={col_id:'article_id',col_title:'title',col_out:'outlet',col_cat:'category',col_dt:'dt'})
df['dt'] = pd.to_datetime(df['dt'], errors='coerce')
df = df.dropna(subset=['title','dt']).reset_index(drop=True)

CATEGORIES = ['International','Politique','Économie']   # same 3 hard-news desks as the paper

if USE_FULL_POOL:
    # (1) restrict to the paper's 3 categories
    b=len(df); df=df[df.category.isin(CATEGORIES)].reset_index(drop=True)
    print(f'category filter: {b:,} -> {len(df):,} (kept {CATEGORIES})')
    # (2) restrict to the 10-outlet panel (pool is the ~18-outlet superset)
    present=set(df.outlet.unique()); matched=[o for o in PANEL_OUTLETS if o in present]
    miss=[o for o in PANEL_OUTLETS if o not in present]
    if miss: print('WARNING: panel outlets not found (name mismatch?):', miss)
    assert len(matched)>=4, f'only {len(matched)} panel outlets found; check spelling vs {sorted(present)}'
    df=df[df.outlet.isin(matched)].reset_index(drop=True)
    print(f'restricted to {len(matched)} panel outlets')
    # (3) apply the SAME headline cleaning as phase-1 clean_pool (pool titles are RAW)
    import re as _re
    _PANEL_DOMAINS={'Le Monde':['lemonde','le monde'],'Le Figaro':['lefigaro','le figaro','figaro'],
        'Les Echos':['lesechos','les echos','les Échos'],'La Croix':['la-croix','la croix'],
        'Ouest-France':['ouest-france','ouest france'],'Le Parisien':['leparisien','le parisien'],
        'TF1 INFO':['tf1info','tf1 info','tf1','lci'],'Libération':['liberation','libération','libe','libé'],
        'Franceinfo':['franceinfo','france info','francetvinfo'],'BFMTV':['bfmtv','bfm tv','bfm']}
    def normalize_ws(s):
        if not isinstance(s,str): return ''
        s=html.unescape(s)
        s=''.join(ch for ch in s if unicodedata.category(ch)[0]!='C')
        s=_re.sub(r'\blrm\b',' ',s).replace('\xa0',' ')
        return _re.sub(r'\s+',' ',s).strip()
    _SUF=_re.compile(r'\s*[-|–—]\s*('+'|'.join(sorted((_re.escape(v) for vs in _PANEL_DOMAINS.values() for v in vs),key=len,reverse=True))+r')\s*$', _re.I)
    _LEAD=_re.compile(r'^\s*(EN\s+DIRECT|DIRECT|VID[EÉ]O|REPLAY|LIVE|PHOTOS?|EN\s+IMAGES|INFOGRAPHIES?|CARTE|PODCAST|[EÉ]DITO|TRIBUNE|D[EÉ]CRYPTAGE|REPORTAGE|T[EÉ]MOIGNAGE|ENTRETIEN|INTERVIEW)\s*[:.\-–—]\s+', _re.I)
    _IDX=_re.compile(r'(?i)(toute l.actualit|actualit[eé]s?\s*,\s*infos|donn[eé]es pays|infos? et actualit|tous les articles|derni[eè]res actualit|fil info|toute l.info)')
    def clean_headline(s):
        s=normalize_ws(s); s=_re.sub(r'https?://\S+|www\.\S+',' ',s)
        prev=None
        while prev!=s: prev=s; s=_SUF.sub('',s)
        s=_LEAD.sub('',s); s=_re.sub(r'\s+',' ',s).strip()
        return _re.sub(r'^[\-|–—:.\s]+|[\s\-|–—]+$','',s).strip()
    n0=len(df)
    df['title']=df['title'].map(clean_headline)
    df=df[df['title'].astype(bool) & ~df['title'].map(lambda s: bool(_IDX.search(str(s))))]
    df=df[(df['title'].str.len()>=10) & (df['title'].str.split().map(len)>=2)]
    df['_h']=df['title'].map(lambda t: hashlib.md5(t.encode('utf-8')).hexdigest())
    df=df.drop_duplicates('_h').drop_duplicates(['outlet','title']).drop(columns='_h').reset_index(drop=True)
    print(f'headline cleaning (phase-1 clean_pool): {n0:,} -> {len(df):,}')
# (the balanced sampled_dataset is already cleaned + category-restricted by phase 1, so no-op there)

df = df[['article_id','title','outlet','category','dt']].copy()
LABELS = sorted(df.outlet.unique()); CHANCE = 1/len(LABELS); CATS = sorted(df.category.unique())
print(f'{len(df):,} headlines | {len(LABELS)} outlets | chance={CHANCE:.3f} | {df.dt.min().date()}..{df.dt.max().date()}')
print(df.outlet.value_counts())

## 3 · Embed headlines

In [ ]:
from sentence_transformers import SentenceTransformer
# INCREMENTAL embedding cache keyed by article_id: encode ONLY new articles, append, reuse the rest.
# Scales to the full pool and survives corpus expansion (same idea as the parse/valence caches).
EMB_NPY = os.path.join(EDITORIAL_DIR, f'analysis_{PANEL}', 'emb_cache.npy')
EMB_IDS = os.path.join(EDITORIAL_DIR, f'analysis_{PANEL}', 'emb_cache_ids.npy')
cache_emb=None; cache_ids=[]
if (not RECOMPUTE_EMB) and os.path.exists(EMB_NPY) and os.path.exists(EMB_IDS):
    cache_emb=np.load(EMB_NPY); cache_ids=list(np.load(EMB_IDS, allow_pickle=True))
    print(f'embedding cache: {len(cache_ids):,} vectors at {EMB_NPY}')
have=set(cache_ids)
new_ids=list(dict.fromkeys(a for a in df.article_id if a not in have))   # order-preserving, de-duped
print(f'{len(df)-len([a for a in df.article_id if a in have]):,} reused | {len(new_ids):,} NEW to embed')
if new_ids:
    enc=SentenceTransformer(ENCODER)
    sub=df[df.article_id.isin(set(new_ids))].drop_duplicates('article_id').set_index('article_id').loc[new_ids]
    vecs=enc.encode(sub.title.tolist(), batch_size=128, show_progress_bar=True,
                    convert_to_numpy=True, normalize_embeddings=True)
    cache_emb = vecs if cache_emb is None else np.vstack([cache_emb, vecs])
    cache_ids = cache_ids + new_ids
    os.makedirs(os.path.dirname(EMB_NPY), exist_ok=True)
    np.save(EMB_NPY, cache_emb); np.save(EMB_IDS, np.array(cache_ids, dtype=object))
    print(f'SAVED embedding cache -> {EMB_NPY} | total {len(cache_ids):,}')
id2row={a:i for i,a in enumerate(cache_ids)}
emb = cache_emb[[id2row[a] for a in df.article_id]]   # aligned to df row order
print('emb aligned to corpus:', emb.shape)

## 4 · spaCy + NER for clustering and measures
Load once; used by NER-overlap (4b), attribution (dep parse), and agency/voice.

In [ ]:
import spacy, unicodedata, re
from collections import Counter
try:
    nlp = spacy.load('fr_core_news_lg')
except OSError:
    from spacy.cli import download; download('fr_core_news_lg'); nlp = spacy.load('fr_core_news_lg')
print('spaCy', spacy.__version__, '| pipes:', nlp.pipe_names)

# ============================================================================
# Framing extractors — SYNCED to tier2_gold_codebook.md v2 (2026-06-23).
# Any change to a rule here MUST be mirrored in the codebook and the RULES_VERSION
# bumped (the parse cache auto-invalidates on a version change). See codebook §2-§5.
# ============================================================================
RULES_VERSION = '2026-06-23-codebook-v2.2'

def _norm(t): return ''.join(c for c in unicodedata.normalize('NFKD', str(t).lower()) if not unicodedata.combining(c))
def _l(tok): return _norm(tok.lemma_)
def _has(tok, attr, val):
    try: return val in tok.morph.get(attr)
    except Exception: return False

# --- lexicons (all accent-stripped lemmas) ---
# SAY-class: report a proposition -> attribution ONLY when governing a ccomp clause or a quote (codebook §2 rule 3)
SAY = {'affirmer','declarer','dire','estimer','juger','assurer','soutenir','expliquer','indiquer','preciser',
       'prevenir','confirmer','dementir','reconnaitre','repondre','retorquer','repliquer','marteler','insinuer',
       'suggerer','annoncer','promettre','prevoir'}
# STANCE-class: voiced position -> attribution when ACTIVE with an explicit subject (noun object OK) (codebook §2 rule 4)
# NB: 'condamner' deliberately EXCLUDED (dominantly judicial/passive in this corpus -> handled as action).
STANCE = {'denoncer','accuser','fustiger','saluer','regretter','deplorer','alerter','plaider','prevenir','avertir','juger','estimer'}
# pure-report verbs: report propositional content even with an infinitival (xcomp) complement (codebook §2 r3)
REPORT = {'dire','affirmer','declarer','estimer','juger','assurer','soutenir','expliquer','reconnaitre',
          'dementir','confirmer','indiquer','preciser','pretendre','repondre','retorquer','repliquer',
          'marteler','insinuer','suggerer'}
# surface forms (accent-stripped) of reporting verbs — robust fallback when spaCy mis-tags POS/lemma
REPORTING_SURFACE = {'estime','estiment','affirme','affirment','declare','declarent','annonce','annoncent',
          'assure','assurent','juge','jugent','soutient','soutiennent','explique','expliquent','indique',
          'indiquent','precise','pretend','denonce','denoncent','accuse','accusent','fustige','salue','saluent',
          'regrette','deplore','alerte','plaide','previent','promet','promettent','repond','retorque','replique',
          'martele','insinue','suggere','dit','disent','reconnait','dement','confirme','confirment','prevoit'}
# epistemic hedge lemmas / words (codebook §5)
HEDGE_LEMMAS = {'pouvoir','sembler','paraitre','risquer'}
HEDGE_WORDS  = {'peut-etre','probablement'}   # 'sans doute' handled in text scan
# unaccusative / happening verbs: subject is an undergoer, never an agent; their participles are NOT passive
UNACCUSATIVE = {'etre','deceder','mourir','exister','rester','devenir','arriver','tomber','grimper','chuter',
                'baisser','augmenter','monter','survenir','apparaitre','disparaitre','naitre','exploser',
                'seffondrer','effondrer','progresser','reculer','stagner','culminer','flamber','plonger',
                'derailler','seproduire'}
# common-noun / institutional actors that count as agents (codebook §4 ACTOR-NOUN list)
ACTOR_NOUNS = {'gouvernement','etat','ministere','ministre','president','presidente','maire','prefet','parquet',
               'justice','tribunal','police','gendarmerie','armee','syndicat','patronat','parti','opposition',
               'majorite','entreprise','groupe','societe','banque','direction','conseil','commission','assemblee',
               'senat','parlement','union','club','equipe','federation','association','ong','eglise','cathedrale',
               'manifestant','salarie','greviste','electeur','habitant','depute','senateur','elu','dirigeant',
               'autorite','autorites','etats-unis','etat-unis'}
QUOTE_RE = re.compile(r'[«"“”»]\s*(.+?)\s*[«"“”»]')
EVID_RE  = re.compile(r"\b(selon|d['’]?apres)\b")
POURINIT_RE = re.compile(r"^pour\s+[A-ZÀ-Ý]")

def feats_from_doc(doc):
    txt  = doc.text
    low  = _norm(txt)
    ents = sorted({e.text.lower() for e in doc.ents if e.label_ in ('PER','LOC','ORG','MISC')})
    ent_acts = {e.text.lower() for e in doc.ents if e.label_ in ('PER','ORG')}

    qspans = QUOTE_RE.findall(txt)
    qlens = [len(t.split()) for t in qspans]
    short_quote = any(1 <= n <= 4 for n in qlens)    # scare/contested-term quote -> hedging (codebook §5 r3)
    quote_finite = False                             # quote is reported SPEECH iff it contains a finite verb
    for mt in QUOTE_RE.finditer(txt):
        a, b = mt.start(1), mt.end(1)
        if any(_has(tok,'VerbForm','Fin') for tok in doc if a <= tok.idx < b):
            quote_finite = True; break

    # ---- ATTRIBUTION (codebook §2) ----
    evidential = bool(EVID_RE.search(low)) or bool(POURINIT_RE.match(txt.strip()))
    def _stance_ok(t):   # 'saluer la memoire/mort' = tribute idiom, NOT a voiced stance (codebook §2; FP #12)
        return _l(t) in STANCE and not (_l(t)=='saluer'
               and any(_l(c) in ('memoire','mort','disparition','souvenir') for c in t.children))
    # any reporting cue at all? (lemma OR surface form OR selon) -> needed for a quote to count as attribution
    reporting_cue = (evidential or any(_l(t) in (SAY|STANCE) for t in doc)
                     or any(_norm(t.text) in REPORTING_SURFACE for t in doc))
    # SAY/REPORT + complement clause: 'que/qu' complementizer (dep-independent), ccomp, OR infinitival xcomp (REPORT verbs)
    say_clause = any(
        (_l(t) in SAY and (any(c.dep_=='ccomp' for c in t.children)
                           or any(_norm(tk.text) in ('que','qu') for tk in doc if t.i < tk.i <= t.i+3)))
        or (_l(t) in REPORT and any(c.dep_ in ('ccomp','xcomp') and c.pos_ in ('VERB','AUX') for c in t.children))
        for t in doc)
    # surface 'verb + que' fallback (catches POS-mistagged reporting verbs, e.g. 'estime que')
    say_que_surface = any(_norm(t.text) in REPORTING_SURFACE
                          and any(_norm(tk.text) in ('que','qu') for tk in doc if t.i < tk.i <= t.i+3)
                          for t in doc)
    # inverted quotative ("..., annonce/juge X"): reporting verb after comma/closing-quote, capitalized source after
    inverted = False
    for t in doc:
        if _norm(t.text) in REPORTING_SURFACE or _l(t) in (SAY|STANCE):
            prev = doc[t.i-1] if t.i>0 else None
            if prev is not None and (prev.is_punct or prev.text in (',','»','"','“','”',':')):
                if any(doc[j].is_alpha and doc[j].text[:1].isupper()
                       for j in range(t.i+1, min(len(doc), t.i+4))):
                    inverted = True; break
    inverted = inverted or any(_l(t) in (SAY|STANCE)
                               and any(c.dep_=='nsubj' and c.i>t.i for c in t.children) for t in doc)
    stance_active = any(_stance_ok(t) and not _has(t,'Voice','Pass')
                        and any(c.dep_=='nsubj' for c in t.children) for t in doc)
    sedire = any(_l(t) in ('dire','declarer')
                 and any((_l(c) in ('se','s') or c.dep_ in ('expl','expl:subj','expl:comp')) for c in t.children)
                 and any(c.pos_=='ADJ' for c in t.children) for t in doc)
    quote_attr = quote_finite and reporting_cue      # quote counts ONLY with a speaker/reporting cue (codebook §2 r2)
    attribution = int(evidential or say_clause or say_que_surface or inverted or stance_active or sedire or quote_attr)

    # ---- PASSIVE (codebook §3) ----
    passive = 0
    for t in doc:
        if t.dep_ in ('aux:pass','nsubj:pass') or _has(t,'Voice','Pass'):
            passive = 1; break
        # pronominal passive: "se voit infliger ..."
        if _l(t) == 'voir' and any(_l(c) in ('se','s') for c in t.children) and any(c.dep_=='xcomp' for c in t.children):
            passive = 1; break
        # reduced participial passive: participle predicated of its subject, no 'avoir' aux,
        # not an etre+intransitive perfect, not reflexive
        if _has(t,'VerbForm','Part') and t.pos_ in ('VERB','ADJ'):
            if _l(t) in UNACCUSATIVE:            # est mort / est decede / est arrive -> NOT passive
                continue
            kids = list(t.children)
            has_avoir = any(_l(c)=='avoir' and c.dep_.startswith('aux') for c in kids)
            reflexive = any(_l(c) in ('se','s') for c in kids)
            if has_avoir or reflexive:
                continue
            has_par_agent = any(c.dep_ in ('obl:agent','obl:mod') and _norm(c.text)!='' for c in kids) \
                            or bool(re.search(r'\bpar\b', _norm(' '.join(c.text for c in kids))))
            subj_child = any(c.dep_ in ('nsubj','nsubj:pass') for c in kids)
            if has_par_agent or subj_child or t.dep_ == 'ROOT' or t.dep_ == 'acl' or t.head.dep_ == 'ROOT':
                passive = 1; break

    verbless = int(not any(t.pos_=='VERB' and t.dep_=='ROOT' for t in doc))

    # ---- AGENT_SUBJECT (codebook §4): named OR common-noun/collective actor as ACTIVE subject ----
    agent_subject = 0
    if not passive:
        for t in doc:
            if t.dep_ != 'nsubj':
                continue
            h = t.head
            if _has(h,'Voice','Pass') or _l(h) in UNACCUSATIVE:
                continue
            tl = t.text.lower()
            is_actor = (t.pos_ == 'PROPN'
                        or tl in ent_acts
                        or _l(t) in ACTOR_NOUNS
                        or (t.pos_ in ('NOUN','PROPN') and _has(t,'Number','Plur') and t.text[:1].isupper()))
            if is_actor:
                agent_subject = 1; break

    # ---- HEDGING (codebook §5) ----
    hedging = int(bool(EVID_RE.search(low))                      # selon / d'apres  (also attribution)
                  or 'sans doute' in low
                  or any(_l(t) in HEDGE_LEMMAS for t in doc)
                  or any(_norm(t.text) in HEDGE_WORDS for t in doc)
                  or any(_has(t,'Mood','Cnd') for t in doc)       # conditional: serait / aurait
                  or short_quote)                                 # short scare quote only

    return dict(ents=';'.join(ents), attribution=attribution, hedging=hedging,
                passive=passive, agent_subject=agent_subject, verbless=verbless)


# ---- incremental Drive cache keyed by article_id; AUTO-INVALIDATES on RULES_VERSION change ----
# (diagnosis item 4: the old cache was keyed on article_id only, so rule/cleaning changes silently
#  reused stale features. The rules_version stamp forces a full re-parse whenever the logic changes.)
CACHE = os.path.join(EDITORIAL_DIR, f'analysis_{PANEL}', 'parse_cache_lg.parquet')
FEAT_COLS = ['article_id','ents','attribution','hedging','passive','agent_subject','verbless']
COLS = FEAT_COLS + ['rules_version']
if os.path.exists(CACHE):
    cache = pd.read_parquet(CACHE)
    if 'rules_version' not in cache.columns or (cache['rules_version'].astype(str) != RULES_VERSION).any():
        print(f'parse cache STALE (rules -> {RULES_VERSION}); REBUILDING from scratch')
        cache = pd.DataFrame(columns=COLS)
else:
    cache = pd.DataFrame(columns=COLS)
print(f'parse cache: {len(cache):,} articles cached at {CACHE}' if len(cache) else 'parse cache: empty / rebuilding')
todo = df[~df.article_id.isin(set(cache.article_id))][['article_id','title']].reset_index(drop=True)
print(f'{len(df)-len(todo):,} reused from cache | {len(todo):,} NEW articles to parse')
if len(todo):
    try: from tqdm.auto import tqdm
    except Exception: tqdm=lambda x,**k: x
    ids=todo.article_id.tolist(); titles=todo.title.tolist(); new=[]
    for k,doc in enumerate(tqdm(nlp.pipe(titles, batch_size=256), total=len(titles), desc='spaCy parse')):
        f=feats_from_doc(doc); f['article_id']=ids[k]; f['rules_version']=RULES_VERSION; new.append(f)
        if (k+1)%5000==0: print(f'  ...parsed {k+1:,}/{len(titles):,}', flush=True)
    cache=pd.concat([cache, pd.DataFrame(new)[COLS]], ignore_index=True).drop_duplicates('article_id', keep='last')
    os.makedirs(os.path.dirname(CACHE), exist_ok=True); cache.to_parquet(CACHE)
    print(f'SAVED parse cache -> {CACHE} | total {len(cache):,}')
else:
    print('nothing new to parse - fully served from cache')

# ---- attach cached features to df (left-merge preserves row order -> stays aligned with emb) ----
df = df.drop(columns=[c for c in FEAT_COLS if c!='article_id' and c in df.columns] + (['_ents'] if '_ents' in df.columns else []),
             errors='ignore')
df = df.merge(cache[FEAT_COLS], on='article_id', how='left')
df['_ents'] = df['ents'].fillna('').map(lambda x: set(x.split(';')) if x else set())
for col in ['attribution','hedging','passive','agent_subject','verbless']:
    df[col] = df[col].fillna(0).astype(int)
df['char_len'] = df.title.str.len()
print('attached: ents + attribution(%.3f) hedging(%.3f) passive(%.3f) agent(%.3f) verbless(%.3f)'
      % (df.attribution.mean(), df.hedging.mean(), df.passive.mean(), df.agent_subject.mean(), df.verbless.mean()))


## 4b · Calibrate cosine threshold on labeled same/different-event pairs
Build a pair packet for hand-labeling, then (if labels present) pick the operating point at
same-event precision >= 0.90. Until labeled, COS_THRESHOLD from config is used and this prints guidance.

In [ ]:
from sentence_transformers import util
import torch
PAIR_PACKET = os.path.join(OUT, 'tier2_pair_packet.csv')
PAIR_LABELED = os.path.join(OUT, 'tier2_pair_labeled.csv')

if not os.path.exists(PAIR_PACKET):
    rng = np.random.RandomState(0); rows=[]
    embT = torch.tensor(emb)
    for _ in range(400):
        i = rng.randint(len(df))
        # candidate same-category, within-window neighbour
        m = (df.category==df.category.iloc[i]) & (abs((df.dt-df.dt.iloc[i]).dt.days)<=DATE_WINDOW) & (df.index!=i)
        cand = np.where(m.values)[0]
        if len(cand)==0: continue
        sims = util.cos_sim(embT[i], embT[cand])[0].numpy()
        j = cand[int(np.argmax(sims))]
        rows.append(dict(i=i, j=int(j), cos=float(sims.max()),
                         title_i=df.title.iloc[i], title_j=df.title.iloc[j],
                         same_event=''))   # <- the author fills 1/0
    pd.DataFrame(rows).to_csv(PAIR_PACKET, index=False)
    print('WROTE pair packet for labeling ->', PAIR_PACKET, '| fill same_event (1/0), save as', PAIR_LABELED)

if os.path.exists(PAIR_LABELED):
    pl = pd.read_csv(PAIR_LABELED).dropna(subset=['same_event'])
    from sklearn.metrics import precision_recall_curve
    p,r,th = precision_recall_curve(pl.same_event.astype(int), pl.cos)
    ok = np.where(p[:-1] >= 0.90)[0]
    if len(ok):
        COS_THRESHOLD = float(th[ok[0]])
        print(f'CALIBRATED COS_THRESHOLD={COS_THRESHOLD:.3f} (same-event precision>=0.90, recall={r[ok[0]]:.2f})')
    else:
        print('no threshold reaches precision 0.90 on labeled pairs - inspect packet')
else:
    print(f'pair labels not found; using config COS_THRESHOLD={COS_THRESHOLD}')

## 5 · Build matched-event clusters (NER-constrained, Leiden/HDBSCAN, with purity)
Within `category x date-window`, build a similarity graph with edges where cos>=threshold AND >=NER_OVERLAP_MIN
shared entities; partition with Leiden (fallback: greedy components). Keep clusters with >=MIN_OUTLETS outlets.

In [ ]:
from sentence_transformers import util
import torch
from collections import Counter
# TIME-BLOCKED clustering: per (category, calendar-month) run community_detection on that block only, so each
# call stays small/fast/low-memory even on the full pool. Then enforce date span + NER overlap + >=MIN_OUTLETS.
# (Events straddling a month boundary may split; minor power loss, no bias.)
rows=[]; cid=0
df['_ym']=df.dt.dt.to_period('M').astype(str)
for cat in CATS:
    for blk in sorted(df.loc[df.category==cat,'_ym'].unique()):
        idx=df.index[(df.category==cat)&(df._ym==blk)].to_numpy()
        if len(idx)<MIN_OUTLETS: continue
        E=torch.tensor(emb[idx])
        comms=util.community_detection(E, threshold=COS_THRESHOLD, min_community_size=MIN_OUTLETS)
        for com in comms:
            rr=idx[np.array(com)]; sub=df.loc[rr]
            if (sub.dt.max()-sub.dt.min()).days > DATE_WINDOW: continue
            members=list(rr)
            if NER_OVERLAP_MIN>0:
                ec=Counter()
                for r in rr: ec.update(df.at[r,'_ents'])
                ev={e for e,c in ec.items() if c>=2}
                if ev: members=[r for r in rr if len(df.at[r,'_ents'] & ev) >= NER_OVERLAP_MIN]
            if df.loc[members].outlet.nunique() >= MIN_OUTLETS:
                for r in members: rows.append((cid, r, cat))
                cid+=1
cl=pd.DataFrame(rows, columns=['cluster','row','cat']).set_index('row')
clustered=df.join(cl, how='inner')

nout=clustered.groupby('cluster').outlet.nunique(); sz=clustered.groupby('cluster').size()
print(f'clusters={clustered.cluster.nunique():,} | headlines={len(clustered):,} ({len(clustered)/len(df):.1%} of corpus)')
print('cluster size:', sz.describe()[['mean','min','max']].round(1).to_dict(),
      '| outlets/cluster:', nout.describe()[['mean','min','max']].round(1).to_dict())
print(f'clusters containing ALL {len(LABELS)} outlets: {int((nout==len(LABELS)).sum())} / {len(nout)}  (high share => over-merged blobs)')
clustered[['article_id','title','outlet','category','dt','cluster']].to_parquet(os.path.join(OUT,'clusters_v2.parquet'))

## 5b · Cluster purity (hand-check ~30 clusters)
Sample clusters, dump headlines for an eyeball pass; compute pairwise same-event purity once labeled.

In [ ]:
rng=np.random.RandomState(1)
samp = rng.choice(clustered.cluster.unique(), min(30, clustered.cluster.nunique()), replace=False)
rows=[]
for c in samp:
    sub=clustered[clustered.cluster==c]
    for _,r in sub.iterrows():
        rows.append(dict(cluster=c, outlet=r['outlet'], dt=r['dt'].date(), title=r['title'], same_event=''))
pur = pd.DataFrame(rows); pur.to_csv(os.path.join(OUT,'tier2_purity_packet.csv'), index=False)
print('WROTE purity packet (30 clusters) -> mark same_event per row (majority label per cluster), then:')
print('  purity = mean over clusters of (largest same-event group size choose 2)/(cluster size choose 2)')
for c in samp[:4]:
    print('\n--- cluster', c, '---')
    for t in clustered[clustered.cluster==c].title.head(6): print('  ', t[:90])

## 6 · Gold validation packet for the framing MEASURES (Part 1a)
~250 headlines stratified by outlet x category, oversampling clustered headlines. the author annotates 4 axes.

In [ ]:
GOLD_PACKET=os.path.join(OUT,'tier2_gold_packet.csv')
GOLD_LABELED=os.path.join(OUT,'tier2_gold_annotated.csv')
if not os.path.exists(GOLD_PACKET):
    rng=np.random.RandomState(2)
    in_clusters=set(clustered.index)
    pool=df.copy(); pool['in_cluster']=pool.index.isin(in_clusters)
    # stratify: 2/3 from clustered headlines, balanced over outlet x category
    picks=[]
    per_cell=max(1, 250//(len(LABELS)*len(CATS)))
    for o in LABELS:
        for cat in CATS:
            cell=pool[(pool.outlet==o)&(pool.category==cat)]
            cc=cell[cell.in_cluster]
            take=pd.concat([cc.sample(min(len(cc), per_cell*2//3+1), random_state=2),
                            cell.sample(min(len(cell), per_cell), random_state=2)]).drop_duplicates('article_id')
            picks.append(take.head(per_cell))
    gp=pd.concat(picks).drop_duplicates('article_id').sample(frac=1, random_state=2)
    gp=gp[['article_id','outlet','category','title']].copy()
    for col in ['gold_valence','gold_attribution','gold_passive','gold_agent_subject','gold_hedging']:
        gp[col]=''   # valence in {-1,0,1}; others binary 1/0
    gp.to_csv(GOLD_PACKET, index=False)
    print(f'WROTE gold packet ({len(gp)} headlines) -> {GOLD_PACKET}. Annotate, save as {GOLD_LABELED}.')
else:
    print('gold packet exists:', GOLD_PACKET)

## 7 · Framing measures — **upgraded, non-LLM**

### 7a · Valence — validated ensemble (replaces product-review model)
Aggregate transformer + lexicon backends; report inter-backend agreement and (if gold present) vs-gold rho.

In [ ]:
from transformers import pipeline
import torch
DEVICE = 0 if torch.cuda.is_available() else -1

# incremental Drive cache keyed by article_id: score ONLY new articles (same pattern as the parse cache)
VCACHE = os.path.join(EDITORIAL_DIR, f'analysis_{PANEL}', 'valence_cache.parquet')
vc = pd.read_parquet(VCACHE) if os.path.exists(VCACHE) else pd.DataFrame(columns=['article_id','v_model'])
todo = df[~df.article_id.isin(set(vc.article_id))][['article_id','title']].reset_index(drop=True)
print(f'valence cache: {len(vc):,} cached | {len(todo):,} NEW to score')
if len(todo):
    senti = pipeline('sentiment-analysis', model='cardiffnlp/twitter-xlm-roberta-base-sentiment',
                     device=DEVICE, truncation=True)
    try: from tqdm.auto import tqdm
    except Exception: tqdm=lambda x,**k: x
    ids=todo.article_id.tolist(); vals=[]
    for k,r in enumerate(tqdm(senti(todo.title.tolist(), batch_size=64), total=len(todo), desc='sentiment')):
        lab=r['label'].lower(); sc=r['score']
        vals.append(sc if 'pos' in lab else (-sc if 'neg' in lab else 0.0))   # robust to Positive/POS/LABEL_x
        if (k+1)%5000==0: print(f'  ...scored {k+1:,}/{len(todo):,}', flush=True)
    vc=pd.concat([vc, pd.DataFrame({'article_id':ids,'v_model':vals})], ignore_index=True).drop_duplicates('article_id', keep='last')
    os.makedirs(os.path.dirname(VCACHE), exist_ok=True); vc.to_parquet(VCACHE); print('SAVED valence cache ->', VCACHE)
else:
    print('nothing new to score - fully served from cache')

df = df.drop(columns=[c for c in ['v_model','valence'] if c in df.columns], errors='ignore').merge(vc, on='article_id', how='left')
def z(x): s=np.std(x); return (x-np.mean(x))/s if s>0 else x*0
# valence = z(transformer score). A validated FR lexicon (FEEL/Polarimots) can be blended later once it clears the gold gate.
df['valence'] = z(df['v_model'].fillna(0).values)
print('valence attached (transformer, cached). mean=%.3f sd=%.3f'%(df.valence.mean(), df.valence.std()))

### 7b · Attribution & hedging — dependency parse (replaces substring lists)

In [ ]:
# attribution + hedging were computed once in the cached parse step (cell 4) and are already on df.
print('attribution/hedging (from parse cache):', df[['attribution','hedging']].mean().round(3).to_dict())

### 7c · Agency / voice — UD passive + agent/patient subject

In [ ]:
# agency/voice (passive, agent_subject, verbless) + char_len were computed in the cached parse step (cell 4).
print('agency/voice (from parse cache):', df[['passive','agent_subject','verbless']].mean().round(3).to_dict())

### 7d · Validate measures vs gold (Part 1b) — admission gate

In [ ]:
ADMITTED={}
if os.path.exists(GOLD_LABELED):
    gl=pd.read_csv(GOLD_LABELED)
    gl=gl.merge(df[['article_id','valence','attribution','passive','agent_subject','hedging']], on='article_id', how='left')
    from sklearn.metrics import cohen_kappa_score
    from scipy.stats import spearmanr
    rep=[]
    # continuous: valence
    g=gl.dropna(subset=['gold_valence'])
    rho=spearmanr(g.gold_valence.astype(float), g.valence).correlation if len(g)>5 else np.nan
    rep.append(('valence','rho',rho, rho>=GATE_RHO)); ADMITTED['valence']= (rho>=GATE_RHO)
    # binary measures
    for m,gc in [('attribution','gold_attribution'),('passive','gold_passive'),
                 ('agent_subject','gold_agent_subject'),('hedging','gold_hedging')]:
        g=gl.dropna(subset=[gc])
        k=cohen_kappa_score(g[gc].astype(int), g[m].astype(int)) if len(g)>5 else np.nan
        rep.append((m,'kappa',k, k>=GATE_KAPPA)); ADMITTED[m]=(k>=GATE_KAPPA)
    V=pd.DataFrame(rep, columns=['measure','metric','value','admitted']); V.to_csv(os.path.join(OUT,'tier2_measure_validation.csv'), index=False)
    print(V.to_string(index=False))
    print('\nADMITTED to confirmatory family:', [m for m,a in ADMITTED.items() if a])
    print('INSTRUMENT-LIMITED (null uninterpretable):', [m for m,a in ADMITTED.items() if not a])
else:
    print('gold not annotated yet -> measures provisional; ALL nulls uninterpretable until validated.')
    ADMITTED={m:None for m in ['valence','attribution','passive','agent_subject','hedging']}

## 8 · Assemble per-(cluster, outlet) framing table

In [ ]:
FRAMING=['valence','attribution','agent_subject','passive','verbless','hedging']
# clustered already carries the parse features (attached to df before clustering); drop stale copies, then
# re-attach the CURRENT df features (incl. valence, which is added after the clustering cell) to avoid _x/_y collisions.
clustered=clustered.drop(columns=[c for c in FRAMING+['char_len'] if c in clustered.columns], errors='ignore')
clustered=clustered.merge(df[['article_id']+FRAMING+['char_len']], on='article_id', how='left')
agg=clustered.groupby(['cluster','outlet'])[FRAMING+['char_len']].mean().reset_index()
# deviation from cluster mean = the matched-event contrast
cl_mean=clustered.groupby('cluster')[FRAMING+['char_len']].mean()
dev=agg.copy()
for m in FRAMING+['char_len']:
    dev[m+'_dev']=agg[m].values - cl_mean.loc[agg.cluster, m].values
print('per-(cluster,outlet) rows:', len(agg), '| clusters:', agg.cluster.nunique())

## 9 · Power analysis (Part 2) — events/outlet needed for 80% power

In [ ]:
from statsmodels.stats.power import TTestPower
PILOT_EFF={'valence':0.26,'attribution':0.27}  # observed paired effect sizes
analysis=TTestPower(); rows=[]
for o in LABELS:
    n_o=agg[agg.outlet==o].cluster.nunique()
    for m,eff in PILOT_EFF.items():
        need=analysis.solve_power(effect_size=eff, alpha=0.05/len(CONFIRMATORY), power=0.80, alternative='larger')
        mdes=analysis.solve_power(nobs=n_o, alpha=0.05/len(CONFIRMATORY), power=0.80, alternative='larger') if n_o>3 else float('nan')
        rows.append(dict(outlet=o, measure=m, n_events=n_o, n_needed=round(need), powered=n_o>=need,
                         min_detectable_eff=round(mdes,3) if mdes==mdes else None))
P=pd.DataFrame(rows); P.to_csv(os.path.join(OUT,'tier2_power_analysis.csv'), index=False)
print(P.to_string(index=False))
print('\nUNDERPOWERED outlet x measure (null NOT claimable):',
      P[~P.powered][['outlet','measure']].apply(tuple,axis=1).tolist())

## 10 · Test (a) — predict outlet from event-centered framing alone (+ permutation null)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
X=dev[[m+'_dev' for m in FRAMING]].values.astype(float); y=dev.outlet.values; groups=dev.cluster.values
mask=np.isfinite(X).all(axis=1)
if (~mask).sum(): print(f'dropping {(~mask).sum()} of {len(X)} rows with NaN/inf in framing features')
X,y,groups=X[mask],y[mask],groups[mask]
def cv_acc(X,y,groups):
    nsplit=max(2, min(5, len(np.unique(groups))))
    gk=GroupKFold(nsplit); acc=[]
    for tr,te in gk.split(X,y,groups):
        sc=StandardScaler().fit(X[tr]); clf=LogisticRegression(max_iter=2000,class_weight='balanced')
        clf.fit(sc.transform(X[tr]),y[tr]); acc.append((clf.predict(sc.transform(X[te]))==y[te]).mean())
    return np.mean(acc)
obs=cv_acc(X,y,groups)
rng=np.random.RandomState(0); null=[]
for _ in range(200):
    yp=rng.permutation(y); null.append(cv_acc(X,yp,groups))
null=np.array(null)
print(f'framing-only attribution acc={obs:.3f} | chance={CHANCE:.3f} | perm-null={null.mean():.3f}+/-{null.std():.3f}'
      f' | p={(null>=obs).mean():.3f}')

## 11 · Test (b) — mixed-effects per measure (Part 3b)
`measure ~ C(outlet) + char_len + (1|cluster)`. Outlet fixed effects vs a reference; cluster random intercept =
the matched-event control. Far more powerful than 60 per-cell Wilcoxon tests.

In [ ]:
import statsmodels.api as sm
# WITHIN-EVENT fixed-effects estimator, built WITHOUT patsy formulas (patsy's C()/Sum get shadowed by the
# `C` DataFrame defined later -> the PatsyError you saw). Demean each measure within its cluster (controls for
# the event), build an explicit sum-to-zero (deviation) outlet design, OLS with EVENT-CLUSTERED robust SEs.
recs=[]
for m in FRAMING:
    d=agg[['cluster','outlet',m,'char_len']].dropna().copy().rename(columns={m:'y'})
    if d.y.nunique()<2 or d.outlet.nunique()<3: continue
    outs=sorted(d.outlet.unique())
    ref = REF_OUTLET if REF_OUTLET in outs else outs[-1]      # aliased (deviation = -sum of the rest)
    coded=[o for o in outs if o!=ref]
    d['y_dev']    = d.groupby('cluster')['y'].transform(lambda v: v - v.mean())
    d['clen_dev'] = d.groupby('cluster')['char_len'].transform(lambda v: v - v.mean())
    if d['y_dev'].abs().sum()==0:
        print('no within-event variation for', m, '(skipped)'); continue
    # sum-to-zero coding: col_o = +1 if outlet==o, -1 if outlet==ref, else 0  -> beta_o = deviation vs grand mean
    Xc={}
    ov=d.outlet.values
    for o in coded:
        Xc['outlet_'+o]=np.where(ov==o,1.0,np.where(ov==ref,-1.0,0.0))
    X=pd.DataFrame(Xc, index=d.index); X['clen_dev']=d['clen_dev'].values; X=sm.add_constant(X)
    try:
        mod=sm.OLS(d['y_dev'].values, X).fit(cov_type='cluster', cov_kwds={'groups': d['cluster'].values})
        got={}
        for o in coded:
            nm='outlet_'+o
            got[o]=dict(measure=m, outlet=o, beta=mod.params[nm], se=mod.bse[nm], z=mod.tvalues[nm], p=mod.pvalues[nm])
        recs+=list(got.values())
        recs.append(dict(measure=m, outlet=ref, beta=-sum(v['beta'] for v in got.values()), se=np.nan, z=np.nan, p=np.nan))
    except Exception as e:
        print('within-event OLS failed for', m, '->', repr(e))
ME=pd.DataFrame(recs)
print('fitted', ME.measure.nunique() if len(ME) else 0, 'measures;', len(ME),
      'outlet offsets (within-event FE, cluster-robust SE; REF=%s)'%REF_OUTLET)
print('confirmatory outlets present per measure:',
      ME[ME.outlet.isin([o for o,_,_ in CONFIRMATORY])].groupby('measure').outlet.apply(list).to_dict() if len(ME) else {})
ME.to_csv(os.path.join(OUT,'tier2_mixedeffects.csv'), index=False)

## 12 · Confirmatory family vs exploratory FDR scan (Part 3b)

In [ ]:
from statsmodels.stats.multitest import multipletests
# CONFIRMATORY: only admitted measures; one-sided; Bonferroni over the small family
conf=[]
for outlet,measure,direction in CONFIRMATORY:
    if ADMITTED.get(measure) is False:
        conf.append(dict(outlet=outlet,measure=measure,note='instrument-limited (excluded)')); continue
    row=ME[(ME.outlet==outlet)&(ME.measure==measure)]
    if len(row)==0:
        conf.append(dict(outlet=outlet,measure=measure,note='outlet absent from this measure fit')); continue
    r=row.iloc[0]
    if pd.isna(r.p):
        conf.append(dict(outlet=outlet,measure=measure,beta=round(r.beta,4),
                         note='this outlet is REF_OUTLET (aliased, no Wald p) - set REF_OUTLET to a different non-confirmatory outlet')); continue
    p1=r.p/2 if ((r.beta<0)==(direction=='lt')) else 1-r.p/2
    conf.append(dict(outlet=outlet,measure=measure,direction=direction,beta=round(r.beta,4),
                     p_onesided=round(p1,4), p_bonf=round(min(1,p1*len(CONFIRMATORY)),4),
                     sig=p1*len(CONFIRMATORY)<0.05))
CONF=pd.DataFrame(conf); print('=== CONFIRMATORY (pre-specified, Bonferroni x%d) ==='%len(CONFIRMATORY)); print(CONF.to_string(index=False))

# EXPLORATORY: everything else, FDR (BH)
conf_keys={(o,m) for o,m,_ in CONFIRMATORY}
expl=ME[~ME.set_index(['outlet','measure']).index.isin(conf_keys)].copy() if len(ME) else ME.copy()
expl=expl.dropna(subset=['p']).copy()   # REF_OUTLET rows carry NaN p -> would break BH-FDR
if len(expl):
    expl['p_fdr']=multipletests(expl.p, method='fdr_bh')[1]; expl['sig_fdr']=expl.p_fdr<0.05
    expl=expl.sort_values('p')
    print('\n=== EXPLORATORY (FDR-BH), top 10 ===' ); print(expl.head(10).to_string(index=False))
CONF.to_csv(os.path.join(OUT,'tier2_confirmatory.csv'), index=False); expl.to_csv(os.path.join(OUT,'tier2_exploratory.csv'), index=False)

## 13 · Offset heatmap

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
piv=ME.pivot_table(index='outlet', columns='measure', values='beta')
plt.figure(figsize=(9,5)); sns.heatmap(piv, center=0, cmap='RdBu_r', annot=True, fmt='.2f', cbar_kws={'label':'mixed-effects beta'})
plt.title('Per-outlet framing offsets on matched events (mixed-effects)'); plt.tight_layout()
plt.savefig(os.path.join(OUT,'tier2_v2_offsets_heatmap.png'), dpi=140); plt.show()

## 14 · ⭐ Verdict — judged against the pre-specified numbers

In [ ]:
print('='*74)
print('TIER 2 HARDENED VERDICT'); print('='*74)
print('Instruments admitted (gold-validated):', [m for m,a in ADMITTED.items() if a])
print('Instrument-limited (nulls uninterpretable):', [m for m,a in ADMITTED.items() if a is False])
print('Measures still provisional (no gold yet):', [m for m,a in ADMITTED.items() if a is None])
print('-'*74)
print(f'(a) framing-only attribution: acc={obs:.3f} vs perm-null {null.mean():.3f} (p={(null>=obs).mean():.3f})')
try:
    nsig=int(CONF.sig.sum()); print(f'(b) confirmatory family: {nsig}/{len(CONF)} pre-specified contrasts significant (Bonferroni)')
    print(CONF[['outlet','measure','beta','p_bonf']].to_string(index=False))
except Exception: pass
print('-'*74)
print('Inclusion reading (per TIER2_DESIGN.md s6):')
print('  framing paragraph EARNED if >=1 admitted confirmatory contrast survives Bonferroni AND is powered;')
print('  otherwise report an INSTRUMENT-AWARE bounding null (state which measures were instrument-limited).')

## 15 · How to share back
Sync to repo `tier2/` (overwrite stale): `tier2_mixedeffects.csv`, `tier2_confirmatory.csv`,
`tier2_exploratory.csv`, `tier2_measure_validation.csv`, `tier2_power_analysis.csv`, `tier2_v2_offsets_heatmap.png`,
`clusters_v2.parquet`. Paste the verdict cell output here for the paper-decision call.
Order of operations: (1) label `tier2_pair_packet.csv` -> threshold; (2) annotate `tier2_gold_packet.csv` -> measures;
(3) label `tier2_purity_packet.csv` -> purity; (4) flip USE_FULL_POOL=True for the powered run.

## 16 · Direct gold validation (reads features from `df`, no re-parse)

In [ ]:
# ============================================================================
# §16 · DIRECT GOLD VALIDATION  — uses the features ALREADY in `df` (no re-parse).
# Runs at the end of the full pipeline: validates the EXACT features the analysis
# uses (cell 10 binary measures + cell 21 valence), against the gold codebook v2.
# ============================================================================
import os, numpy as np, pandas as pd
from sklearn.metrics import cohen_kappa_score
from scipy.stats import spearmanr

gpath = os.path.join(OUT, 'tier2_gold_annotated.csv')
if not os.path.exists(gpath):
    gpath = os.path.join(OUT, 'tier2_gold_packet.csv')
    print('NOTE: using tier2_gold_packet.csv — ensure its labels are coded under codebook v2.')
gold = pd.read_csv(gpath)

FEATS    = ['attribution','passive','agent_subject','hedging']
have_val = 'valence' in df.columns
cols     = ['article_id'] + FEATS + (['valence'] if have_val else [])
gv = gold.merge(df[cols], on='article_id', how='left', suffixes=('','_pred'))
matched = int(gv[FEATS[0]].notna().sum())
print(f'gold {len(gold)} rows | matched in df {matched} | unmatched (dropped by cleaning) {len(gold)-matched}')
gv = gv.dropna(subset=FEATS)

GATE_KAPPA = globals().get('GATE_KAPPA', 0.60); GATE_RHO = globals().get('GATE_RHO', 0.50)
rows=[]
if have_val and 'gold_valence' in gv.columns:
    sv = gv.dropna(subset=['gold_valence','valence'])
    rho = spearmanr(sv.gold_valence.astype(float), sv.valence.astype(float)).correlation
    rows.append(('valence','rho',round(rho,3), bool(rho>=GATE_RHO), None,None,None,None))
for m in FEATS:
    sub = gv.dropna(subset=['gold_'+m]); y, yh = sub['gold_'+m].astype(int), sub[m].astype(int)
    k = cohen_kappa_score(y, yh)
    tp=int(((y==1)&(yh==1)).sum()); fp=int(((y==0)&(yh==1)).sum())
    fn=int(((y==1)&(yh==0)).sum()); tn=int(((y==0)&(yh==0)).sum())
    rows.append((m,'kappa',round(k,3), bool(k>=GATE_KAPPA), tp,fp,fn,tn))
V = pd.DataFrame(rows, columns=['measure','metric','value','admitted','TP','FP','FN','TN'])
V.to_csv(os.path.join(OUT,'tier2_measure_validation.csv'), index=False)
print('='*74); print(V.to_string(index=False)); print('='*74)
print('ADMITTED:', list(V[V.admitted].measure), '| FAILS:', list(V[~V.admitted].measure))

# per-measure disagreement counts + full dump for FOCUS
FOCUS='attribution'
for m in FEATS:
    bad = gv[gv['gold_'+m].notna() & (gv['gold_'+m].astype(int) != gv[m].astype(int))]
    print(f'  {m:14} disagreements: {len(bad)}')
bad = gv[gv['gold_'+FOCUS].notna() & (gv['gold_'+FOCUS].astype(int) != gv[FOCUS].astype(int))]
print(f'\n######## {FOCUS}: {len(bad)} disagreements ########')
for _, r in bad.iterrows():
    tag = 'FP' if int(r[FOCUS])==1 else 'FN'
    print(f'[{tag}] {str(r.title)[:95]}')


## STEP 2 · Truncation robustness (self-contained, cache-only, ~0 GPU)

Reloads `clustered` + cached parse/valence features **from disk** (clusters_v2.parquet, parse_cache_lg.parquet,
valence_cache.parquet) — so it does **not** need the pipeline live in the kernel. Drops the ~25-30% truncated
meta-titles, **re-enforces MIN_OUTLETS**, and re-runs the within-event FE estimator (cell 35) + confirmatory
Bonferroni (cell 37). Question: does **BFMTV agent_subject** survive dropping truncated headlines?


In [ ]:
# ============================================================================
# STEP 2 · TRUNCATION ROBUSTNESS  — SELF-CONTAINED (loads from disk; no kernel state needed).
# Mirrors cell 35 (within-event FE, sum-to-zero, cluster-robust SE) and
# cell 37 (one-sided Bonferroni confirmatory). No GPU / no re-parse / no re-embed.
# ============================================================================
import os, sys, glob, numpy as np, pandas as pd, statsmodels.api as sm

# --- locate dirs the same way as cell 4 (works whether or not the pipeline ran) ---
EDITORIAL_DIR = globals().get('EDITORIAL_DIR', os.environ.get('EDITORIAL_DIR', '.'))
PANEL = globals().get('PANEL','n10')
ADIR  = os.path.join(EDITORIAL_DIR, f'analysis_{PANEL}')
OUT   = globals().get('OUT', os.path.join(ADIR, 'tier2_v2'))

def _find(*cands):
    for c in cands:
        hits = glob.glob(c, recursive=True)
        if hits: return hits[0]
    return None

CLUST_F = _find(os.path.join(OUT,'clusters_v2.parquet'),
                os.path.join(ADIR,'**','clusters_v2.parquet'),
                os.path.join(EDITORIAL_DIR,'**','clusters_v2.parquet'))
PARSE_F = _find(os.path.join(ADIR,'parse_cache_lg.parquet'),
                os.path.join(EDITORIAL_DIR,'**','parse_cache_lg.parquet'))
VAL_F   = _find(os.path.join(ADIR,'valence_cache.parquet'),
                os.path.join(EDITORIAL_DIR,'**','valence_cache.parquet'))
assert CLUST_F and PARSE_F and VAL_F, f'missing a cache file: clusters={CLUST_F} parse={PARSE_F} valence={VAL_F}'
print('clusters:', CLUST_F); print('parse   :', PARSE_F); print('valence :', VAL_F)

# --- config (same constants as cell 4) ---
FRAMING     = ['valence','attribution','agent_subject','passive','verbless','hedging']
REF_OUTLET  = globals().get('REF_OUTLET','Le Monde')
MIN_OUTLETS = globals().get('MIN_OUTLETS',4)
CONFIRMATORY = globals().get('CONFIRMATORY', [
    ('Libération','valence','lt'), ('Les Echos','attribution','lt'),
    ('BFMTV','attribution','gt'),  ('BFMTV','agent_subject','gt')])

# ADMITTED from the gold gate: prefer the saved validation table, else default all-admitted (gate passed v2.2)
if 'ADMITTED' in globals():
    ADMITTED = globals()['ADMITTED']
else:
    vpath = _find(os.path.join(OUT,'tier2_measure_validation.csv'), os.path.join(ADIR,'**','tier2_measure_validation.csv'))
    if vpath:
        v = pd.read_csv(vpath); ADMITTED = dict(zip(v.measure, v.admitted.astype(bool)))
    else:
        ADMITTED = {m: True for m in FRAMING}
    print('ADMITTED:', ADMITTED)

# --- rebuild `clustered` + features from disk ---
clustered = pd.read_parquet(CLUST_F)                       # article_id,title,outlet,category,dt,cluster
parse = pd.read_parquet(PARSE_F)
if 'rules_version' in parse.columns:
    rv = sorted(parse.rules_version.dropna().unique())
    assert any('codebook-v2.2' in str(x) for x in rv), f'parse cache is not v2.2: {rv}'
need = ['attribution','passive','agent_subject','verbless','hedging']  # char_len not cached -> derive below
miss = [c for c in need if c not in parse.columns]
assert not miss, f'parse cache missing {miss}; has {list(parse.columns)}'
val = pd.read_parquet(VAL_F)[['article_id','v_model']]

clustered = clustered.merge(parse[['article_id']+need], on='article_id', how='left')
clustered = clustered.merge(val, on='article_id', how='left')
clustered['char_len'] = clustered['title'].str.len()   # cell-10 definition
# valence = z(v_model). Within-event FE demeans valence per cluster, so a global affine z is p-value-invariant.
v = clustered['v_model'].fillna(0).values; sd = v.std()
clustered['valence'] = (v - v.mean())/sd if sd>0 else v*0
cov = clustered[need[:-1]+['valence']].notna().all(axis=1).mean()
print(f'feature coverage of clustered set: {cov:.1%} | rows {len(clustered):,} | clusters {clustered.cluster.nunique():,}')
assert cov > 0.95, 'low feature coverage -> cache/cluster ID mismatch; stop and inspect.'

# --- cell-35 estimator (wrapped) ---
def fit_within_event(clust):
    a = clust.groupby(['cluster','outlet'])[FRAMING+['char_len']].mean().reset_index()
    recs=[]
    for m in FRAMING:
        d=a[['cluster','outlet',m,'char_len']].dropna().copy().rename(columns={m:'y'})
        if d.y.nunique()<2 or d.outlet.nunique()<3: continue
        outs=sorted(d.outlet.unique()); ref=REF_OUTLET if REF_OUTLET in outs else outs[-1]
        coded=[o for o in outs if o!=ref]
        d['y_dev']    = d.groupby('cluster')['y'].transform(lambda x: x-x.mean())
        d['clen_dev'] = d.groupby('cluster')['char_len'].transform(lambda x: x-x.mean())
        if d['y_dev'].abs().sum()==0: continue
        ov=d.outlet.values
        Xc={'outlet_'+o: np.where(ov==o,1.0,np.where(ov==ref,-1.0,0.0)) for o in coded}
        X=pd.DataFrame(Xc,index=d.index); X['clen_dev']=d['clen_dev'].values; X=sm.add_constant(X)
        try:
            mod=sm.OLS(d['y_dev'].values, X).fit(cov_type='cluster', cov_kwds={'groups': d['cluster'].values})
            got=[dict(measure=m,outlet=o,beta=mod.params['outlet_'+o],se=mod.bse['outlet_'+o],
                      z=mod.tvalues['outlet_'+o],p=mod.pvalues['outlet_'+o]) for o in coded]
            recs+=got
            recs.append(dict(measure=m,outlet=ref,beta=-sum(g['beta'] for g in got),se=np.nan,z=np.nan,p=np.nan))
        except Exception as e:
            print('within-event OLS failed for', m, '->', repr(e))
    return pd.DataFrame(recs), a.cluster.nunique()

# --- cell-37 confirmatory family ---
def confirmatory(ME):
    out=[]
    for outlet,measure,direction in CONFIRMATORY:
        if ADMITTED.get(measure) is False:
            out.append(dict(outlet=outlet,measure=measure,note='instrument-limited (excluded)')); continue
        row=ME[(ME.outlet==outlet)&(ME.measure==measure)]
        if len(row)==0:
            out.append(dict(outlet=outlet,measure=measure,note='outlet absent from this measure fit')); continue
        r=row.iloc[0]
        if pd.isna(r.p):
            out.append(dict(outlet=outlet,measure=measure,beta=round(r.beta,4),note='REF_OUTLET (aliased, no Wald p)')); continue
        p1 = r.p/2 if ((r.beta<0)==(direction=='lt')) else 1-r.p/2
        out.append(dict(outlet=outlet,measure=measure,direction=direction,beta=round(r.beta,4),
                        p_onesided=round(p1,4), p_bonf=round(min(1,p1*len(CONFIRMATORY)),4),
                        sig=p1*len(CONFIRMATORY)<0.05))
    return pd.DataFrame(out)

# --- truncation detector (CMS-truncated meta-titles end in an ellipsis; survives phase-1 cleaning) ---
def _is_trunc(t):
    s=str(t).rstrip(); return s.endswith(('…','...','[…]','[...]','(...)','(…)'))
trunc_ids = set(clustered.loc[clustered.title.map(_is_trunc),'article_id'])
print(f'truncated headlines in clustered set: {len(trunc_ids):,}/{len(clustered):,} = {len(trunc_ids)/len(clustered):.1%}')

# (1) full (should reproduce tier2_confirmatory.csv)
ME_full,_ = fit_within_event(clustered); CONF_full = confirmatory(ME_full)
# (2) drop truncated, then RE-ENFORCE MIN_OUTLETS so matched events stay valid
cn = clustered[~clustered.article_id.isin(trunc_ids)].copy()
keep = cn.groupby('cluster').outlet.nunique(); keep = keep[keep>=MIN_OUTLETS].index
cn = cn[cn.cluster.isin(keep)].copy()
print(f'after dropping truncated + re-enforcing >={MIN_OUTLETS} outlets/cluster: '
      f'{clustered.cluster.nunique():,} -> {cn.cluster.nunique():,} clusters | {len(clustered):,} -> {len(cn):,} headlines')
ME_nt,_ = fit_within_event(cn); CONF_nt = confirmatory(ME_nt)

print('\n=== CONFIRMATORY · full pool (all headlines) ===');   print(CONF_full.to_string(index=False))
print('\n=== CONFIRMATORY · truncated headlines DROPPED ==='); print(CONF_nt.to_string(index=False))
print('\n--- result tracking: BFMTV agent_subject (the surviving contrast) ---')
for tag,ME in [('full',ME_full),('no-trunc',ME_nt)]:
    r=ME[(ME.outlet=='BFMTV')&(ME.measure=='agent_subject')]
    if len(r): print(f'  {tag:8s}: beta={r.iloc[0].beta:+.4f}  p={r.iloc[0].p:.4f}')

ME_nt.to_csv(os.path.join(OUT,'tier2_mixedeffects_notrunc.csv'), index=False)
CONF_nt.to_csv(os.path.join(OUT,'tier2_confirmatory_notrunc.csv'), index=False)
print('\nwrote -> tier2_mixedeffects_notrunc.csv , tier2_confirmatory_notrunc.csv  (in OUT)')
print('VERDICT: BFMTV-agency is truncation-robust IFF its sig holds in the no-trunc column above.')
